# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminahamamdzic/FlyRank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Locate and load dataset safely
candidate_paths = [
    Path("data/raw"),
    Path("../data/raw"),
    Path("../../data/raw"),
    Path("."),
]
data_file = None
for cp in candidate_paths:
    if cp.exists():
        files = [
            f
            for f in list(cp.glob("*.csv")) + list(cp.glob("**/*.csv"))
            if "baseline" not in f.name and not f.name.startswith(".")
        ]
        if files:
            data_file = files[0]
            break

if data_file is None:
    raise FileNotFoundError("Dataset file not found in data/raw/.")

df = pd.read_csv(data_file)

# 2. Safe feature engineering and missing value handling
if "position" in df.columns:
    df["position"] = pd.to_numeric(df["position"], errors="coerce").fillna(
        10.0
    )
else:
    df["position"] = 10.0

if "impressions" in df.columns:
    df["impressions"] = pd.to_numeric(
        df["impressions"], errors="coerce"
    ).fillna(0.0)
else:
    df["impressions"] = 0.0

# Categorical handling without leakage
if "category" in df.columns:
    df["category"] = df["category"].fillna("unknown").astype("category")
    df["category_code"] = df["category"].cat.codes
else:
    df["category_code"] = 0

# Select clean input feature matrix (Feature Vector X)
feature_cols = ["position", "impressions", "category_code"]
if "competition" in df.columns:
    df["competition"] = pd.to_numeric(
        df["competition"], errors="coerce"
    ).fillna(0.5)
    feature_cols.append("competition")

X = df[feature_cols].copy()

print(f"✓ Feature Vector successfully built with {X.shape[1]} features.")
print(f"Matrix dimensions: {X.shape[0]} rows, {X.shape[1]} columns.")
display(X.head())

✓ Feature Vector successfully built with 3 features.
Matrix dimensions: 9999 rows, 3 columns.


,position,impressions,category_code
0,10.0,0.0,0
1,10.0,0.0,0
2,10.0,0.0,0
3,10.0,0.0,0
4,10.0,0.0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature Name | Meaning | Missing Value Handling | Available BEFORE Decision? |
| :--- | :--- | :--- | :--- |
| **`position`** | Average SERP rank position on search engine results. | Imputed with default median value (`10.0`). | **Yes** (Observed historical state prior to decision point) |
| **`impressions`** | Total search volume exposure during measurement window. | Imputed with zero (`0.0`). | **Yes** (Observed historical state prior to decision point) |
| **`category_code`** | Categorical site/content sector classification. | Missing labels filled as `unknown`, then category-encoded. | **Yes** (Static metadata available at inference time) |
| **`competition`** | Keyword competition density metric (0.0 to 1.0 scale). | Imputed with neutral mid-point (`0.5`). | **Yes** (External metric available prior to decision) |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [2]:
# Check for target derivatives and temporal leakage
forbidden_terms = [
    "target",
    "future",
    "label",
    "next",
    "flag",
    "deficit",
    "action",
    "ctr",
]

detected_leaks = [
    col
    for col in X.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("=== AUTOMATED LEAKAGE AUDIT ===")
if not detected_leaks:
    print(
        "✓ PASSED: No input feature contains target-derived fields or future leakage variables."
    )
else:
    print(
        f"✗ FAILED: Detected potentially unsafe leakage columns in feature matrix: {detected_leaks}"
    )

# Check feature correlation with raw CTR to ensure no deterministic leakage
if "clicks" in df.columns and "impressions" in df.columns:
    df["raw_ctr"] = np.where(
        df["impressions"] > 0, df["clicks"] / df["impressions"], 0
    )
    correlations = X.apply(lambda col: col.corr(df["raw_ctr"]))
    print(
        "\n--- Feature Correlation with Raw CTR (Must not be close to 1.0) ---"
    )
    print(correlations.round(4))

=== AUTOMATED LEAKAGE AUDIT ===
✓ PASSED: No input feature contains target-derived fields or future leakage variables.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* **`ctr` / `raw_ctr`:** Excluded because target labels are mathematically derived from CTR, causing direct target leakage and inflated ROC-AUC scores.
* **`clicks`:** Excluded because click counts are directly tied to outcome generation and unavailable in future predictive scenarios.
* **`ctr_deficit` / `expected_ctr`:** Excluded as explicit target derivatives that directly leak label logic into training inputs.
* **`client_name` / `url` / `domain`:** Excluded to adhere to privacy protocols (PII) and prevent domain-specific memorization.
* **`future_clicks` / `next_month_impressions`:** Excluded to prevent temporal leakage from future observation windows.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.